In [11]:
import sys
sys.path.append("/home/user/문서/workspace/python/src")

from data_load_save import *


In [12]:
csv_file = f"/home/user/문서/workspace/python/data/콩_무역_2023.csv"

df = pd.read_csv(csv_file)

df.head()

,importer,exporter,value
0,"China, mainland",Brazil,71592496.82
1,"China, mainland",United States of America,26506551.32
2,Argentina,Paraguay,5829120.00
3,Argentina,Brazil,4081112.10
4,Thailand,Brazil,2750180.24


In [14]:
import pandas as pd
import networkx as nx
import itertools
import community  # python-louvain (import community.community_louvain as community_louvain 라고 쓰기도 함)


# 1) 데이터 불러와서 방향 + 가중 네트워크 만들기
def build_trade_network(csv_path, src_col="exporter", dst_col="importer", w_col="value"):
    df = pd.read_csv(csv_path)
    # 0 이상인 값만 사용 (필요시 필터)
    df = df[df[w_col] > 0].copy()
    
    G = nx.DiGraph()
    for _, row in df.iterrows():
        src = row[src_col]
        dst = row[dst_col]
        w = float(row[w_col])
        # 이미 간선 있으면 weight 누적
        if G.has_edge(src, dst):
            G[src][dst]["weight"] += w
        else:
            G.add_edge(src, dst, weight=w)
    return G


# 2) 기본 네트워크 특성 계산
def compute_basic_measures(G):
    print("=== Basic topology ===")
    N = G.number_of_nodes()
    M = G.number_of_edges()
    print(f"Nodes (N): {N}")
    print(f"Edges (M): {M}")
    
    # Network density (directional)
    density = nx.density(G)
    print(f"Density ρ: {density:.4f}")
    
    # 평균 경로 길이 & 직경: 약하게 연결된 최대 컴포넌트에서 계산 (무향으로 변환)
    if N > 0:
        H = G.to_undirected()
        largest_cc_nodes = max(nx.connected_components(H), key=len)
        H_cc = H.subgraph(largest_cc_nodes).copy()
        
        if H_cc.number_of_nodes() > 1:
            L = nx.average_shortest_path_length(H_cc)
            Dia = nx.diameter(H_cc)
            print(f"Average path length L (on largest component): {L:.4f}")
            print(f"Network diameter Dia (on largest component): {Dia}")
        else:
            print("Largest component has only 1 node, cannot compute L/Dia.")
    
    # Degree / Weighted degree
    print("\n=== Degree / Weighted degree (sample) ===")
    # out-degree, in-degree (weighted)
    out_wdeg = G.out_degree(weight="weight")
    in_wdeg = G.in_degree(weight="weight")
    # 그냥 몇 개만 보기
    for n, w in list(out_wdeg)[:5]:
        print(f"{n}: weighted outdegree = {w}")
    for n, w in list(in_wdeg)[:5]:
        print(f"{n}: weighted indegree = {w}")
    
    # Betweenness centrality (unweighted shortest path 기준)
    print("\n=== Betweenness centrality (top 5) ===")
    bc = nx.betweenness_centrality(G, normalized=True, weight=None)
    for n, v in sorted(bc.items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"{n}: BC = {v:.4f}")
    
    # Closeness centrality (무향 변환 후)
    print("\n=== Closeness centrality (top 5, undirected) ===")
    cc = nx.closeness_centrality(G.to_undirected())
    for n, v in sorted(cc.items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"{n}: CC = {v:.4f}")
    
    # Clustering coefficient (무향)
    print("\n=== Clustering coefficient ===")
    clustering = nx.clustering(G.to_undirected(), weight=None)
    avg_clustering = sum(clustering.values()) / len(clustering) if clustering else 0
    print(f"Average clustering coefficient: {avg_clustering:.4f}")
    
    return {
        "density": density,
        "betweenness": bc,
        "closeness": cc,
        "clustering": clustering,
    }


# 3) 모듈성 Q & 커뮤니티 (Louvain)
def compute_modularity_and_communities(G):
    print("\n=== Louvain community detection & modularity Q ===")
    H = G.to_undirected()
    
    # Louvain partition: node -> community_id
    partition = community.best_partition(H, weight="weight")
    # Modularity Q
    Q = community.modularity(partition, H, weight="weight")
    print(f"Modularity Q: {Q:.4f}")
    
    # 커뮤니티 예시 출력
    communities = {}
    for node, com in partition.items():
        communities.setdefault(com, []).append(node)
    
    print(f"Number of communities: {len(communities)}")
    for cid, nodes in list(communities.items())[:3]:
        print(f"Community {cid}: {nodes[:10]}{'...' if len(nodes) > 10 else ''}")
    
    return partition, Q


# 4) HITS hub / authority
def compute_hits(G, max_iter=1000, tol=1e-08):
    print("\n=== HITS hub / authority (top 5) ===")
    hubs, authorities = nx.hits(G, max_iter=max_iter, tol=tol, normalized=True)
    
    print("Top hubs:")
    for n, v in sorted(hubs.items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"{n}: hub = {v:.4f}")
    
    print("Top authorities:")
    for n, v in sorted(authorities.items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"{n}: auth = {v:.4f}")
    
    return hubs, authorities


# 5) Network efficiency E
def network_efficiency(G):
    """
    E = 1 / [n(n-1)] * sum_{i != j} 1/d_ij
    여기서는 무향 그래프의 최단경로 기준으로 계산
    """
    H = G.to_undirected()
    nodes = list(H.nodes())
    n = len(nodes)
    if n < 2:
        return 0.0
    
    eff_sum = 0.0
    pair_count = 0
    
    for i, j in itertools.combinations(nodes, 2):
        try:
            d = nx.shortest_path_length(H, i, j)
            eff_sum += 1.0 / d
            pair_count += 1
        except nx.NetworkXNoPath:
            # 연결 안되어 있으면 그 쌍은 기여 0으로 처리
            continue
    
    if pair_count == 0:
        return 0.0
    
    # 위에서 i<j 조합만 돌렸으므로 분모는 n(n-1)/2 → 논문식에 맞추면:
    # E = (2 * eff_sum) / [n(n-1)]
    E = (2.0 * eff_sum) / (n * (n - 1))
    return E


def compute_robustness_scenarios(G, remove_sets):
    """
    remove_sets: 예) {"Scenario 1 (remove Brazil)": ["Brazil"], "Scenario 2 (remove USA)": ["USA"]}
    """
    print("\n=== Network efficiency robustness scenarios ===")
    baseline_E = network_efficiency(G)
    print(f"Baseline E: {baseline_E:.4f}")
    
    results = {"Baseline": baseline_E}
    
    for name, nodes_to_remove in remove_sets.items():
        G_copy = G.copy()
        G_copy.remove_nodes_from(nodes_to_remove)
        E_val = network_efficiency(G_copy)
        print(f"{name}: remove {nodes_to_remove} → E = {E_val:.4f}")
        results[name] = E_val
    return results


# 6) 메인 실행 예시
if __name__ == "__main__":
    csv_path = "/home/user/문서/workspace/python/data/콩_무역_2023.csv"  # 경로만 바꿔주면 됨
    
    G = build_trade_network(csv_path, src_col="exporter", dst_col="importer", w_col="value")
    
    measures = compute_basic_measures(G)
    
    partition, Q = compute_modularity_and_communities(G)
    
    hubs, authorities = compute_hits(G)
    
    E0 = network_efficiency(G)
    print(f"\nOverall network efficiency E: {E0:.4f}")
    
    # 예시: 중국, 브라질, 미국이 노드 이름이라고 가정
    robustness_results = compute_robustness_scenarios(
        G,
        {
            "Scenario 1 (remove Brazil)": ["Brazil"],
            "Scenario 2 (remove USA)": ["USA"],
            "Scenario 3 (remove Brazil & USA)": ["Brazil", "USA"],
        },
    )


=== Basic topology ===
Nodes (N): 142
Edges (M): 1000
Density ρ: 0.0499
Average path length L (on largest component): 2.3367
Network diameter Dia (on largest component): 5

=== Degree / Weighted degree (sample) ===
Brazil: weighted outdegree = 91326573.11
China, mainland: weighted outdegree = 63283.17
United States of America: weighted outdegree = 44215092.42
Paraguay: weighted outdegree = 6000665.48
Argentina: weighted outdegree = 2195566.38
Brazil: weighted indegree = 181024.33
China, mainland: weighted indegree = 103383644.16999999
United States of America: weighted indegree = 662987.19
Paraguay: weighted indegree = 9541.02
Argentina: weighted indegree = 10367695.040000001

=== Betweenness centrality (top 5) ===
United States of America: BC = 0.1589
Netherlands (Kingdom of the): BC = 0.0977
France: BC = 0.0893
Canada: BC = 0.0829
China, mainland: BC = 0.0636

=== Closeness centrality (top 5, undirected) ===
China, mainland: CC = 0.6409
United States of America: CC = 0.6380
Canada: C